# Gemma 4 E2B × TinyCeNN — Integrated Memory V1

This is the Gemma-4 version of the working Qwen3.5 integrated-memory Colab. It uses the exact requested base model **`google/gemma-4-E2B`** and benchmarks the **text backbone only**. Vision/audio towers are not modified or evaluated here.

Gemma 4 E2B has seven original full-attention text layers at **4, 9, 14, 19, 24, 29, 34**. Because Gemma 4 shares KV states in its final 20 layers and layer 14 is the full-attention shared-KV producer, V1 safely replaces only independent full-attention layers **4 and 9**. Sliding attention and the shared-KV producer/consumers remain native.

- conservative CeNN: replace layer `4`
- expanded CeNN: replace layers `4,9`
- validation NLL selects only between CeNN candidates
- held-out test documents are not used for selection
- native-relative cache equivalence is checked after training
- final section compares real sample prompts on original Gemma 4 vs selected TinyCeNN


## 1. Setup — clone latest TinyCeNN-LM and verify Gemma 4 architecture


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN detected.')
else:
    print('No HF_TOKEN secret found; public download will be attempted.')

REPO = Path(tempfile.mkdtemp(prefix='gemma4-e2b-cenn-')) / 'TinyCeNN-LM'
subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)

subprocess.run([sys.executable,'-m','pip','install','-q',
                'transformers==5.17.0','huggingface_hub>=0.36.2','datasets>=3,<6',
                'accelerate','pytest','pandas','matplotlib'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'], check=True)

os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO),str(REPO/'src')])
sys.path[:0] = [str(REPO),str(REPO/'src')]

import torch
from transformers import AutoConfig

MODEL_ID = 'google/gemma-4-E2B'
cfg = AutoConfig.from_pretrained(MODEL_ID, token=HF_TOKEN).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t == 'full_attention']
FIRST_SHARED = cfg.num_hidden_layers - cfg.num_kv_shared_layers
NONSHARED_FULL = [i for i in FULL if i < FIRST_SHARED]
SHARED_KV_PRODUCER = max(NONSHARED_FULL)
SAFE_REPLACEABLE = [i for i in NONSHARED_FULL if i != SHARED_KV_PRODUCER]
SOURCE = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()

print('Source:', SOURCE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('GPU VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('Model type:', cfg.model_type)
print('Layers:', cfg.num_hidden_layers)
print('Full attention:', FULL)
print('First KV-shared layer:', FIRST_SHARED)
print('Full-attention shared-KV producer:', SHARED_KV_PRODUCER)
print('TinyCeNN V1 safely replaceable:', SAFE_REPLACEABLE)

assert cfg.model_type == 'gemma4_text'
assert FULL == [4,9,14,19,24,29,34]
assert cfg.num_kv_shared_layers == 20
assert SAFE_REPLACEABLE == [4,9]


## 2. Configuration
`smoke` is the safest first run. The public Gemma-4 E2B checkpoint is much larger than Qwen3.5-0.8B, and training temporarily holds teacher + student.


In [ ]:
PROFILE = 'smoke' # @param ['smoke','balanced','extended']
SAVE_TO_DRIVE = True # @param {type:'boolean'}

PROFILES = {
  'smoke': dict(train_contexts='96,128',test_contexts='96,128,256',block_size=16,features=32,train_documents=4,validation_documents=2,test_documents=2,warm_documents=2,warm_steps=2,joint_steps=4,eval_every=2,timing_documents=1,timing_repeats=1,decode_tokens=8,loss_chunk=2),
  'balanced': dict(train_contexts='128,256',test_contexts='128,256,512,1024',block_size=32,features=64,train_documents=24,validation_documents=6,test_documents=8,warm_documents=4,warm_steps=30,joint_steps=60,eval_every=15,timing_documents=1,timing_repeats=1,decode_tokens=24,loss_chunk=4),
  'extended': dict(train_contexts='128,256,512',test_contexts='128,256,512,1024,2048',block_size=32,features=96,train_documents=48,validation_documents=10,test_documents=16,warm_documents=8,warm_steps=60,joint_steps=120,eval_every=20,timing_documents=2,timing_repeats=2,decode_tokens=32,loss_chunk=4),
}

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime for Gemma 4 E2B.')
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
if PROFILE != 'smoke' and VRAM_GB < 30:
    print('⚠️ Balanced/extended may exceed this GPU memory. Start with smoke or use an A100-class runtime.')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/TinyCeNN/gemma4-e2b-integrated-v1')
else:
    BASE = Path('/content/gemma4-e2b-integrated-v1')
BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = BASE / run_id
LOG = BASE / (run_id + '.log')
RUN = dict(PROFILES[PROFILE], seed=2041)
print(json.dumps(RUN, indent=2))
print('Results:', OUT)


## 3. Preflight — Gemma 4 adapter, KV-sharing protection, cache regression


In [ ]:
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)
SOURCE = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Testing source:', SOURCE)
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
r = subprocess.run([sys.executable,'-m','pytest','-q','tests/test_gemma4_integrated_memory.py'],
                   cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode:
    raise RuntimeError(f'Gemma 4 preflight failed: {r.returncode}')
BENCHMARK = REPO / 'scripts/benchmark_gemma4_e2b_integrated_memory.py'
probe = subprocess.run([sys.executable,str(BENCHMARK),'--help'],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(probe.stdout.splitlines()[0] if probe.stdout else '')
if probe.returncode:
    raise RuntimeError('Gemma 4 benchmark entrypoint failed')
print('✅ Gemma 4 V1 preflight passed')


## 4. Train + evaluate


In [ ]:
cmd = [sys.executable,'-u',str(BENCHMARK),'--base-model',MODEL_ID,'--output-dir',str(OUT)]
for k,v in RUN.items():
    cmd += ['--' + k.replace('_','-'), str(v)]
print('Running:', BENCHMARK.name)
print(' '.join(cmd))
try:
    with LOG.open('w') as log:
        with subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=os.environ.copy()) as p:
            for line in p.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush()
            status = p.wait()
    if status:
        raise RuntimeError(f'Run failed: {status}; inspect {LOG}')
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG,OUT/'console.log')
        print('Archive:',shutil.make_archive(str(OUT)+'-results','zip',root_dir=OUT))


## 5. Results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
s = pd.read_csv(OUT/'integrated_summary.csv')
selected = json.loads((OUT/'selection.json').read_text())['selected']
cols = [c for c in [
    'candidate','context','test_nll','test_perplexity','ppl_ratio','adapted_ppl_ratio',
    'total_cache_ratio','prefill_speedup','decode_speedup','remaining_full_attention_layers',
    'cached_logits_nmse','teacher_cached_logits_nmse','candidate_top1_mismatches',
    'teacher_top1_mismatches','allowed_top1_mismatches','cache_equivalence_passed','selected_on_validation'
] if c in s.columns]
print('Locked validation selection:', selected)
display(s[cols].sort_values(['candidate','context']).reset_index(drop=True))
x = s[s.candidate == selected].sort_values('context')
fig,ax = plt.subplots(figsize=(9,4))
ax.plot(x.context,x.ppl_ratio,marker='o',label='PPL ratio')
ax.plot(x.context,x.total_cache_ratio,marker='s',label='cache ratio')
ax.axhline(1,linestyle='--')
ax.set_xscale('log',base=2); ax.set_xlabel('Context'); ax.legend(); ax.set_title(selected); plt.show()


## 6. Sample user prompts — original Gemma 4 vs validation-selected TinyCeNN

This is a deterministic greedy comparison. `google/gemma-4-E2B` is the requested **base** checkpoint rather than the instruction-tuned `-it` variant, so conversational style is not itself an instruction-following benchmark. The useful comparison is whether the CeNN replacement materially changes the base model's behavior.


In [ ]:
import gc
from transformers import AutoTokenizer, Gemma4ForCausalLM
from transformers.cache_utils import DynamicCache
from tinycenn_lm.gemma4_integrated_memory import restore_student, new_cache, inference_mode, native_dtype

manifest = json.loads((OUT/'manifest.json').read_text())
report = json.loads((OUT/'integrated_report.json').read_text())
SELECTED = json.loads((OUT/'selection.json').read_text())['selected']
record = next(r for r in report['candidates'] if r['candidate'] == SELECTED)
checkpoint = OUT / record['checkpoint']
device = torch.device('cuda')
dtype = native_dtype(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=manifest['model_revision'], token=HF_TOKEN)
original = Gemma4ForCausalLM.from_pretrained(MODEL_ID,revision=manifest['model_revision'],dtype=dtype,attn_implementation='sdpa',token=HF_TOKEN).to(device).eval()

def prompt_ids(prompt):
    try:
        return tokenizer.apply_chat_template([{'role':'user','content':prompt}],add_generation_prompt=True,tokenize=True,return_tensors='pt').to(device)
    except Exception:
        return tokenizer('User: '+prompt+'\nAssistant:',return_tensors='pt').input_ids.to(device)

@torch.no_grad()
def native_generate(model, ids, tokens=48):
    cache = DynamicCache(config=model.config.get_text_config(decoder=True))
    logits = model(input_ids=ids,past_key_values=cache,use_cache=True).logits[:,-1]
    out=[]
    eos = tokenizer.eos_token_id
    eos = {eos} if isinstance(eos,int) else set(eos or [])
    for _ in range(tokens):
        tok=logits.argmax(-1,keepdim=True); out.append(tok)
        if int(tok.item()) in eos: break
        logits=model(input_ids=tok,past_key_values=cache,use_cache=True).logits[:,-1]
    return torch.cat(out,1)

PROMPTS = [
    'Explain in simple words why the sky is blue.',
    'Calculate 18 times 7 and explain the calculation briefly.',
    'Write a short Python function that returns whether a number is prime.',
    'A train travels 120 km in 90 minutes. What is its average speed in km/h?',
    'What are three practical differences between RAM and SSD storage?',
]

original_rows=[]
for prompt in PROMPTS:
    ids=prompt_ids(prompt); toks=native_generate(original,ids)
    original_rows.append((prompt,ids.detach().cpu(),toks.detach().cpu(),tokenizer.decode(toks[0],skip_special_tokens=True).strip()))

payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
cenn=restore_student(original,payload).to(device).eval()
del original; gc.collect(); torch.cuda.empty_cache()

rows=[]
for prompt,ids_cpu,orig_cpu,orig_text in original_rows:
    ids=ids_cpu.to(device)
    eos=tokenizer.eos_token_id; eos_ids=[eos] if isinstance(eos,int) else list(eos or [])
    with inference_mode(cenn,str(dtype).removeprefix('torch.')):
        cenn_tokens,_=__import__('tinycenn_lm.gemma4_integrated_memory',fromlist=['greedy_generate']).greedy_generate(cenn,ids,48,eos_ids)
    cenn_cpu=cenn_tokens.detach().cpu(); common=min(orig_cpu.shape[1],cenn_cpu.shape[1])
    agreement=float((orig_cpu[:,:common]==cenn_cpu[:,:common]).float().mean()) if common else 0.0
    prefix=0
    for a,b in zip(orig_cpu[0].tolist(),cenn_cpu[0].tolist()):
        if a!=b: break
        prefix+=1
    cenn_text=tokenizer.decode(cenn_cpu[0],skip_special_tokens=True).strip()
    print('\n'+'='*100+'\nUSER:',prompt,'\n\nORIGINAL GEMMA 4:\n',orig_text,'\n\nTinyCeNN:\n',cenn_text)
    print(f'\nToken agreement={agreement:.1%}; identical prefix={prefix} tokens')
    rows.append({'prompt':prompt,'original':orig_text,'cenn':cenn_text,'token_agreement':agreement,'identical_prefix_tokens':prefix,'original_tokens':orig_cpu.shape[1],'cenn_tokens':cenn_cpu.shape[1]})
chat=pd.DataFrame(rows); display(chat[['prompt','token_agreement','identical_prefix_tokens','original_tokens','cenn_tokens']])
chat.to_csv(OUT/'chat_comparison.csv',index=False)
(OUT/'chat_comparison.json').write_text(json.dumps(rows,indent=2,ensure_ascii=False))
print('Mean token agreement:',f'{chat.token_agreement.mean():.2%}')
print('✅ Saved chat comparison in',OUT)
